# `rmgdb` and `rmgdatabase`

This demo notebook shows how to use the SQL-wrapped version of RMG-database to access all of the various data contained within.

`standard` holds the `rmgdb` package, which specifies the actual layout of the database.
`data` holds the `rmgdatabase` package, which uses `rmgdb` to build an actual database file from the `RMG-database` Python source files.
`data` also includes a plaintext dump of `rmdatabase` into the YAML format - this is easier to read and more broadly intercompatible with other programming tools than the original Python files in `RMG-database`.
This demo won't cover using these files, but they are available.

There are five sub-databases:

 1. kinetics
 2. solvation
 3. statmech
 4. thermo
 5. transport

Each has libraries (collated data from the literature) and families (RMG-specific subsets from the libraries).

Running this notebook to access the data only requires one dependency: `pandas`.
No database setup is needed; Python includes `sqlite3` in its standard library, which runs without any additional complications.

One could also substitute `pandas` for `polars`, `narwhals`, etc. - really any library that can read from a SQL database.
Enterprising users may also see fit to just use `sqlite3` directly and avoid dependencies altogether, but this requires a more advanced understanding of writing SQL queries.

In [1]:
import pandas as pd

Some quick background to help make this notebook make sense: much of RMG-database stores data as shown below in this random example from the solvation sub-database.

```python
entry(
    index = 2,
    label = "propane",
    molecule = "CCC",
    solute = SoluteData(
        S = 0,
        B = 0,
        E = 0,
        L = 1.05,
        A = 0,
        V = 0.5313,
    ),
    shortDesc = """""",
    longDesc =
"""
From Abarahm et al., J. Chem. Soc., Perkin Trans. 2, 1994, 1777-1791,
DOI: 10.1039/P29940001777
""",
)
```

You can see that the `SoluteData` class is __nested__ inside our call to `entry`.
This is strictly forbidden in SQL databases; instead we store all of the calls to `entry` in one table, all of the calls to `SoluteData` in another, and then provide a 'lookup key' to match the two of them back up.

This process of re-joining the two tables is cumbersome and requires knowing some SQL.
To avoid that, SQL supports __views__ - these are basically just queries against the database that you can treat like regular tables.
They avoid you having to write SQL statements to 'rebuild' the totally flat version of the database.

All of the examples below use the various views to retrieve data.
You can of course directly look at the tables, but there's no need (unless you want to write your own SQL, in which case I suggest familiarizing yourself with the schema in `standard`).
The below function is not needed to actually  use `rmgdatabase`, but is included to help the demo.

In [2]:
import sqlite3

def list_all_views(database_file):
    """
    Connects to an SQLite database and lists all views.
    
    Args:
        database_file (str): The path to the SQLite database file.
    
    Returns:
        list: A list of view names.
    """
    conn = None
    try:
        # Create a database connection
        conn = sqlite3.connect(database_file)
        cursor = conn.cursor()

        # Query the sqlite_master table for views
        cursor.execute("SELECT name FROM sqlite_master WHERE type='view' ORDER BY name;")
        
        # Fetch all results
        views = cursor.fetchall()

        # Print the results
        if views:
            print(f"Views in database '{database_file}':")
            for view in views:
                print(f"- {view[0]}")
        else:
            print(f"No views found in database '{database_file}'.")
        
        # Return the list of view names
        return [view[0] for view in views]

    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        return []
    finally:
        # Close the connection
        if conn:
            conn.close()


One final note - RMG uses its own format for storing molecular structures called the 'adjacency list'.
These can be converted into more friendly formats (INCHI, SMILES) using RMG-Py.

## `transport`

Let's start by looking at what views we have:

In [3]:
transport_db = "data/rmgdatabase/transport/transport.db"
list_all_views(transport_db);


Views in database 'data/rmgdatabase/transport/transport.db':
- label_pairs_view
- transport_groups_view
- transport_libraries_view


The `libraries` view contains data from the literature, digitized into `rmgdatabase`.
The `groups` view contains the actual substructures used by RMG to make estimations for transport properties, with the `label_pairs` view showing how the rows of that table are related to one another in the tree structure.
This demo is focused on just getting data out of `rmgdatabase` - future work can look toward re-building the estimator tree and integrating with RMG-Py.

Let's open up the transport libraries:

In [4]:
pd.read_sql("""SELECT * from transport_libraries_view""", "sqlite:///" + transport_db).set_index("id")

,name,short_description,long_description,label,adjacency_list,shapeIndex,epsilon,epsilon_unit,sigma,sigma_unit,dipoleMoment,dipoleMoment_unit,polarizability,polarizability_unit,rotrelaxcollnum
id,,,,,,,,,,,,,,,
0,GRI-Mech,GRI-Mech3.0 value for AR,,AR,\n1 Ar u0 p4 c0\n,0,1134.930,J/mol,3.330,angstroms,0.0,C*m,0.00,angstroms^3,0.0
1,GRI-Mech,GRI-Mech3.0 value for C,,C(T),\nmultiplicity 3\n1 C u2 p1 c0\n,0,593.655,J/mol,3.298,angstroms,0.0,C*m,0.00,angstroms^3,0.0
2,GRI-Mech,GRI-Mech3.0 value for C2,,C2,"\nmultiplicity 3\n1 C u1 p0 c0 {2,T}\n2 C u1 p...",1,810.913,J/mol,3.621,angstroms,0.0,C*m,1.76,angstroms^3,4.0
3,GRI-Mech,GRI-Mech3.0 value for C2O,\nSame value as C2O(S).\n,C2O(T),"\nmultiplicity 3\n1 C u2 p0 c0 {2,D}\n2 C u0 p...",1,1932.290,J/mol,3.828,angstroms,0.0,C*m,0.00,angstroms^3,1.0
4,GRI-Mech,GRI-Mech3.0 value for C2O,\nSame Value as C2O(T).\n,C2O(S),"\n1 C u0 p1 c0 {2,D}\n2 C u0 p0 c0 {1,D} {3,D}...",1,1932.290,J/mol,3.828,angstroms,0.0,C*m,0.00,angstroms^3,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,NOx2018,,,H2NCHO,"\n 1 N u0 p1 c0 {2,S} {4,S} {5,S}\n 2 C ...",2,307.800,K,4.140,angstroms,0.0,De,0.00,angstroms^3,1.0
383,NOx2018,,,H2NCO,"\n multiplicity 2\n 1 N u0 p1 c0 {2,S} {...",2,307.800,K,4.140,angstroms,0.0,De,0.00,angstroms^3,1.0
384,NOx2018,,,CH3NC,"\n multiplicity 3\n 1 C u0 p0 c0 {2,S} {...",2,422.220,K,5.329,angstroms,3.5,De,0.00,angstroms^3,1.0


We can now see all of the columns that are available - we probably only want to use a subset of these, so here's a function to do so:

In [5]:
def read_sql(view_name, database_file, columns=None):
    return pd.read_sql(f"""SELECT {', '.join(columns) if columns else '*'} from {view_name}""", "sqlite:///" + database_file)

In [6]:
read_sql("transport_libraries_view", transport_db, columns=["id", "name", "adjacency_list", "epsilon", "sigma"]).set_index("id")

,name,adjacency_list,epsilon,sigma
id,,,,
0,GRI-Mech,\n1 Ar u0 p4 c0\n,1134.930,3.330
1,GRI-Mech,\nmultiplicity 3\n1 C u2 p1 c0\n,593.655,3.298
2,GRI-Mech,"\nmultiplicity 3\n1 C u1 p0 c0 {2,T}\n2 C u1 p...",810.913,3.621
3,GRI-Mech,"\nmultiplicity 3\n1 C u2 p0 c0 {2,D}\n2 C u0 p...",1932.290,3.828
4,GRI-Mech,"\n1 C u0 p1 c0 {2,D}\n2 C u0 p0 c0 {1,D} {3,D}...",1932.290,3.828
...,...,...,...,...
382,NOx2018,"\n 1 N u0 p1 c0 {2,S} {4,S} {5,S}\n 2 C ...",307.800,4.140
383,NOx2018,"\n multiplicity 2\n 1 N u0 p1 c0 {2,S} {...",307.800,4.140
384,NOx2018,"\n multiplicity 3\n 1 C u0 p0 c0 {2,S} {...",422.220,5.329


LLMs are generally _very_ good at writing small functions like this, so they are highly recommended for this application.
This demo contains a number of useful functions for loading the sub-databases, as well.

One can also just load the _entire_ view into memory with pandas, and then throw data out as needed, though this may be less efficient.
For example, here's a query that loads only a subset of the columns (renaming one of them, just for fun) with the additional requirement that `epsilon` and `adjacency_list` are present:

In [7]:
pd.read_sql("""
    SELECT name as library_name, adjacency_list, sigma, sigma_unit 
    FROM transport_libraries_view 
    WHERE epsilon IS NOT NULL AND adjacency_list IS NOT NULL
""", "sqlite:///" + transport_db)


,library_name,adjacency_list,sigma,sigma_unit
0,GRI-Mech,\n1 Ar u0 p4 c0\n,3.330,angstroms
1,GRI-Mech,\nmultiplicity 3\n1 C u2 p1 c0\n,3.298,angstroms
2,GRI-Mech,"\nmultiplicity 3\n1 C u1 p0 c0 {2,T}\n2 C u1 p...",3.621,angstroms
3,GRI-Mech,"\nmultiplicity 3\n1 C u2 p0 c0 {2,D}\n2 C u0 p...",3.828,angstroms
4,GRI-Mech,"\n1 C u0 p1 c0 {2,D}\n2 C u0 p0 c0 {1,D} {3,D}...",3.828,angstroms
...,...,...,...,...
382,NOx2018,"\n 1 N u0 p1 c0 {2,S} {4,S} {5,S}\n 2 C ...",4.140,angstroms
383,NOx2018,"\n multiplicity 2\n 1 N u0 p1 c0 {2,S} {...",4.140,angstroms
384,NOx2018,"\n multiplicity 3\n 1 C u0 p0 c0 {2,S} {...",5.329,angstroms
385,NOx2018,"\n multiplicity 2\n 1 C u0 p0 c0 {2,S} {...",4.860,angstroms


And here's the same, but in Pandas:

In [8]:
df_transport_all = pd.read_sql("SELECT * FROM transport_libraries_view", "sqlite:///" + transport_db).set_index("id")
df_transport = df_transport_all[
    df_transport_all['epsilon'].notna() & 
    df_transport_all['adjacency_list'].notna()
].copy()
df_transport.rename(columns={'name': 'library_name'}, inplace=True)
df_transport[['library_name', 'adjacency_list', 'sigma', 'sigma_unit']]

,library_name,adjacency_list,sigma,sigma_unit
id,,,,
0,GRI-Mech,\n1 Ar u0 p4 c0\n,3.330,angstroms
1,GRI-Mech,\nmultiplicity 3\n1 C u2 p1 c0\n,3.298,angstroms
2,GRI-Mech,"\nmultiplicity 3\n1 C u1 p0 c0 {2,T}\n2 C u1 p...",3.621,angstroms
3,GRI-Mech,"\nmultiplicity 3\n1 C u2 p0 c0 {2,D}\n2 C u0 p...",3.828,angstroms
4,GRI-Mech,"\n1 C u0 p1 c0 {2,D}\n2 C u0 p0 c0 {1,D} {3,D}...",3.828,angstroms
...,...,...,...,...
382,NOx2018,"\n 1 N u0 p1 c0 {2,S} {4,S} {5,S}\n 2 C ...",4.140,angstroms
383,NOx2018,"\n multiplicity 2\n 1 N u0 p1 c0 {2,S} {...",4.140,angstroms
384,NOx2018,"\n multiplicity 3\n 1 C u0 p0 c0 {2,S} {...",5.329,angstroms


## `thermo`

Once more, let's look at the views:

In [9]:
thermo_db = "data/rmgdatabase/thermo/thermo.db"
list_all_views(thermo_db);


Views in database 'data/rmgdatabase/thermo/thermo.db':
- label_pairs_view
- thermo_depositories_view
- thermo_groups_view
- thermo_libraries_view


Much the same story as the `transport` sub-database, with the only addition being the `depositories` view (mean for storing metadata, currently unused).

Let's focus on just the `libraries` view, since it is a bit more complicated than `transport`:

In [10]:
thermo_df = read_sql("thermo_libraries_view", thermo_db).set_index("id")
thermo_df.head(2)

,name,short_description,long_description,label,adjacency_list,Tdata_unit,Cpdata_unit,H298,H298_unit,S298,...,c1,c2,c3,c4,c5,c6,c7,poly_Tmin,poly_Tmax,poly_T_unit
id,,,,,,,,,,,,,,,,,,,,,
0,JetSurF1.0,120186,,Ar,\n1 Ar u0 p4 c0\n,NaN,NaN,NaN,NaN,NaN,...,2.5,0.0,0.0,0.0,0.0,-745.375,4.366,298.0,1000.0,K
0,JetSurF1.0,120186,,Ar,\n1 Ar u0 p4 c0\n,NaN,NaN,NaN,NaN,NaN,...,2.5,0.0,0.0,0.0,0.0,-745.375,4.366,1000.0,5000.0,K


In [11]:
print(thermo_df.columns)

Index(['name', 'short_description', 'long_description', 'label',
       'adjacency_list', 'Tdata_unit', 'Cpdata_unit', 'H298', 'H298_unit',
       'S298', 'S298_unit', 'Tdata_1', 'Tdata_2', 'Tdata_3', 'Tdata_4',
       'Tdata_5', 'Tdata_6', 'Tdata_7', 'Cpdata_1', 'Cpdata_2', 'Cpdata_3',
       'Cpdata_4', 'Cpdata_5', 'Cpdata_6', 'Cpdata_7', 'nasa_Tmin',
       'nasa_Tmax', 'nasa_T_unit', 'E0', 'E0_unit', 'Cp0', 'Cp0_unit', 'CpInf',
       'CpInf_unit', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'poly_Tmin',
       'poly_Tmax', 'poly_T_unit'],
      dtype='str')


Generally speaking, records in the thermo library have _either_ experimental data (e.g., H298) _or_ a NASA polynomial (e.g., NASA_Tmax).
We can select each of these in Pandas:

In [12]:
thermo_data_df = thermo_df[thermo_df['H298'].notna() & thermo_df['adjacency_list'].notna()].copy().dropna(axis='columns', how='all')
thermo_data_df.head(2)

,name,short_description,long_description,label,adjacency_list,Tdata_unit,Cpdata_unit,H298,H298_unit,S298,...,Tdata_5,Tdata_6,Tdata_7,Cpdata_1,Cpdata_2,Cpdata_3,Cpdata_4,Cpdata_5,Cpdata_6,Cpdata_7
id,,,,,,,,,,,,,,,,,,,,,
194,BurkeH2O2,Low T polynomial Tmin changed from 300.0 to 29...,\nH 120186 H 1 ...,H,\nmultiplicity 2\n1 H u1 p0 c0\n,K,cal/(mol*K),52.1,kcal/mol,27.39,...,800.0,1000.0,1500.0,4.97,4.97,4.97,4.97,4.97,4.97,4.97
195,BurkeH2O2,Low T polynomial Tmin changed from 300.0 to 29...,\nH2 121286 H 2 ...,H2,"\n1 H u0 p0 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",K,cal/(mol*K),0.0,kcal/mol,31.21,...,800.0,1000.0,1500.0,6.90,6.96,7.00,7.02,7.07,7.21,7.73


There is an additional step for the NASA polynomials: each species can (and usually _does_) have multiple NSA polynomials that are valid at different temperature ranges.
One may wish to simply group these together, or perhaps select only the polynomials valid at a certain temperature.

See the [RMG docs](https://reactionmechanismgenerator.github.io/RMG-Py/reference/thermo/nasa.html) for more information about NASA estimations.

In [13]:
thermo_nasa_df = thermo_df[thermo_df['nasa_Tmin'].notna() & thermo_df['adjacency_list'].notna()].copy().dropna(axis='columns', how='all')
thermo_nasa_df.head(5)

,name,short_description,long_description,label,adjacency_list,nasa_Tmin,nasa_Tmax,nasa_T_unit,E0,E0_unit,...,c1,c2,c3,c4,c5,c6,c7,poly_Tmin,poly_Tmax,poly_T_unit
id,,,,,,,,,,,,,,,,,,,,,
0,JetSurF1.0,120186,,Ar,\n1 Ar u0 p4 c0\n,298.0,5000.0,K,NaN,NaN,...,2.50000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,-745.375,4.366000,298.0,1000.0,K
0,JetSurF1.0,120186,,Ar,\n1 Ar u0 p4 c0\n,298.0,5000.0,K,NaN,NaN,...,2.50000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,-745.375,4.366000,1000.0,5000.0,K
1,JetSurF1.0,121286,,N2,"\n1 N u0 p1 c0 {2,T}\n2 N u0 p1 c0 {1,T}\n",298.0,5000.0,K,NaN,NaN,...,2.92664,0.001488,-5.684760e-07,1.009700e-10,-6.753350e-15,-922.798,5.980530,1000.0,5000.0,K
1,JetSurF1.0,121286,,N2,"\n1 N u0 p1 c0 {2,T}\n2 N u0 p1 c0 {1,T}\n",298.0,5000.0,K,NaN,NaN,...,3.29868,0.001408,-3.963220e-06,5.641520e-09,-2.444850e-12,-1020.900,3.950370,298.0,1000.0,K
2,JetSurF1.0,L10/90,,He,\n1 He u0 p1 c0\n,200.0,6000.0,K,NaN,NaN,...,2.50000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,-745.375,0.928724,200.0,1000.0,K


Use the `id` column for grouping together the multiple polynomials per species:

In [14]:
for id, sub_df in thermo_nasa_df.groupby('id'):
    print(sub_df)
    break

          name short_description long_description label     adjacency_list  \
id                                                                           
0   JetSurF1.0            120186                     Ar  \n1 Ar u0 p4 c0\n   
0   JetSurF1.0            120186                     Ar  \n1 Ar u0 p4 c0\n   

    nasa_Tmin  nasa_Tmax nasa_T_unit  E0 E0_unit  ...   c1   c2   c3   c4  \
id                                                ...                       
0       298.0     5000.0           K NaN     NaN  ...  2.5  0.0  0.0  0.0   
0       298.0     5000.0           K NaN     NaN  ...  2.5  0.0  0.0  0.0   

     c5       c6     c7  poly_Tmin  poly_Tmax  poly_T_unit  
id                                                          
0   0.0 -745.375  4.366      298.0     1000.0            K  
0   0.0 -745.375  4.366     1000.0     5000.0            K  

[2 rows x 24 columns]


Use filters to select at specific temperatures, e.g., between 1000 and 2000 K:

In [15]:
thermo_nasa_df[(thermo_nasa_df['poly_Tmin'] >= 1000) & (thermo_nasa_df['poly_Tmax'] <= 2000)].head(5)

,name,short_description,long_description,label,adjacency_list,nasa_Tmin,nasa_Tmax,nasa_T_unit,E0,E0_unit,...,c1,c2,c3,c4,c5,c6,c7,poly_Tmin,poly_Tmax,poly_T_unit
id,,,,,,,,,,,,,,,,,,,,,
91,JetSurF1.0,T12/89,,C5H5,"\nmultiplicity 2\n1 C u1 p0 c0 {2,S} {5,S} {6...",298.0,2000.0,K,NaN,NaN,...,7.47439,0.016013,-6.482310e-09,-3.581970e-09,9.236510e-13,28086.00,-16.13300,1000.0,2000.0,K
1053,SulfurHaynes,Leeds,,HSO2,"\nmultiplicity 2\n1 S u1 p0 c0 {2,D} {3,D} {4,...",298.0,2000.0,K,NaN,NaN,...,1.56274,0.020691,-2.311210e-05,1.267020e-08,-2.727420e-12,-18214.80,17.55680,1000.0,2000.0,K
1054,SulfurHaynes,Leeds,,HOSO,"\nmultiplicity 2\n1 O u0 p2 c0 {2,S} {4,S}\n2 ...",298.0,2000.0,K,NaN,NaN,...,9.60147,-0.025359,6.768290e-05,-6.349540e-08,1.958940e-11,-31254.00,-15.67410,1000.0,2000.0,K
1056,SulfurHaynes,Leeds,,HOSO2,"\nmultiplicity 2\n1 S u0 p1 c0 {2,S} {3,D} {4,...",298.0,2000.0,K,NaN,NaN,...,7.62277,-0.004199,3.520550e-05,-4.127150e-08,1.400070e-11,-46947.80,-7.80788,1000.0,2000.0,K
1064,SulfurHaynes,,"\nH298 taken from P.A. Denis, Chem. Phys. Lett...",HSO,"\nmultiplicity 2\n1 S u1 p1 c0 {2,S} {3,D}\n2 ...",298.0,2000.0,K,NaN,NaN,...,3.27129,0.005450,-3.737790e-06,1.300210e-09,-1.831140e-13,-3808.55,9.02815,1000.0,2000.0,K


## `solvation`

In [16]:
solvation_db = "data/rmgdatabase/solvation/solvation.db"
list_all_views(solvation_db);

Views in database 'data/rmgdatabase/solvation/solvation.db':
- label_pairs_view
- solute_groups_view
- solute_libraries_view
- solvent_libraries_view


This is also largely the same as the previous sub-databases, except that the libraries are split into two different views: solute and solvent.
There isn't any fundamental architectural reason for this, more just convenience for loading the two separately:

In [17]:
read_sql("solute_libraries_view", solvation_db).set_index("id")

,name,short_description,long_description,label,molecule,S,B,E,L,A,V
id,,,,,,,,,,,
0,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",methane,C,0.000000,0.000000,0.000000,-0.323000,0.000000,0.2495
1,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",ethane,CC,0.000000,0.000000,0.000000,0.492000,0.000000,0.3904
2,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",propane,CCC,0.000000,0.000000,0.000000,1.050000,0.000000,0.5313
3,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",n-butane,CCCC,0.000000,0.000000,0.000000,1.615000,0.000000,0.6722
4,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",2-methylpropane,CC(C)C,0.000000,0.000000,0.000000,1.409000,0.000000,0.6722
...,...,...,...,...,...,...,...,...,...,...,...
445,solute,COSMO fit,\nGeometries from LithiumPrimaryThermo library...,[CH2]C#N,[CH2]C#N,0.722629,0.275263,0.360986,1.656567,0.086384,0.3827
446,solute,COSMO fit,\nGeometries from LithiumPrimaryThermo library...,[Li]N=[C]C,[Li]N=[C]C,1.451317,-0.278358,-1.267313,-2.600696,0.777030,0.5609
447,solute,COSMO fit,\nGeometries from LithiumPrimaryThermo library...,[Li]N=CC,[Li]N=CC,2.391183,1.099293,5.293384,10.955795,1.180529,0.5824


The solvent table has many more columns than the solutes:

In [18]:
df = read_sql("solvent_libraries_view", solvation_db).set_index("id")
df.columns

Index(['name', 'short_description', 'long_description', 'label', 'molecule',
       's_g', 'b_g', 'e_g', 'l_g', 'a_g', 'c_g', 's_h', 'b_h', 'e_h', 'l_h',
       'a_h', 'c_h', 'A', 'B', 'C', 'D', 'E', 'alpha', 'beta', 'eps', 'n',
       'name_in_coolprop', 'dGsolvCount', 'dGsolvMAE_val', 'dGsolvMAE_unit',
       'dHsolvCount', 'dHsolvMAE_val', 'dHsolvMAE_unit'],
      dtype='str')

In [19]:
df.head(5)

,name,short_description,long_description,label,molecule,s_g,b_g,e_g,l_g,a_g,...,beta,eps,n,name_in_coolprop,dGsolvCount,dGsolvMAE_val,dGsolvMAE_unit,dHsolvCount,dHsolvMAE_val,dHsolvMAE_unit
id,,,,,,,,,,,,,,,,,,,,,
0,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,water,O,2.74983,4.84491,0.83346,-0.22544,3.92725,...,0.38,80.4,1.33300,water,5224.0,0.17,kcal/mol,58.0,1.04,kcal/mol
1,solvent,,"\nalpha = 0.328, #primary alcohols\nbeta = 0.4...",1-octanol,CCCCCCCCO,0.71369,1.42785,0.01254,0.85312,3.52275,...,0.45,10.3,1.42050,NaN,4189.0,0.21,kcal/mol,164.0,0.50,kcal/mol
2,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,benzene,C1=CC=CC=C1,1.07490,0.17492,-0.32585,1.01356,0.56683,...,0.14,2.3,1.50110,benzene,110.0,0.19,kcal/mol,200.0,0.35,kcal/mol
3,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,cyclohexane,C1CCCCC1,0.00000,-0.03443,-0.32662,1.03470,0.00000,...,0.00,2.0,1.42662,CycloHexane,122.0,0.22,kcal/mol,226.0,0.31,kcal/mol
4,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,dibutylether,CCCCOCCCC,0.63588,-0.27257,-0.36266,0.98243,2.44885,...,0.45,3.1,1.39920,NaN,90.0,0.20,kcal/mol,78.0,0.27,kcal/mol


All of which can be filtered against, as done previously:

In [20]:
df[df['eps'].notna()][["label", "molecule", "eps"]].head(5)

,label,molecule,eps
id,,,
0,water,O,80.4
1,1-octanol,CCCCCCCCO,10.3
2,benzene,C1=CC=CC=C1,2.3
3,cyclohexane,C1CCCCC1,2.0
4,dibutylether,CCCCOCCCC,3.1


Notice that for these views, in this sub-database only, `rmgdatabase` does not use RMG's adjacency list format.
This is because `RMG-database`, for this data only, natively stores the structures as SMILES.

## `statmech`

Starting with the views:

In [21]:
statmech_db = "data/rmgdatabase/statmech/statmech.db"
list_all_views(statmech_db)

Views in database 'data/rmgdatabase/statmech/statmech.db':
- label_pairs_view
- statmech_groups_view
- statmech_libraries_view


['label_pairs_view', 'statmech_groups_view', 'statmech_libraries_view']

This is another very standard sub-database!

In [22]:
df = read_sql("statmech_libraries_view", statmech_db).set_index("id")
df.head(3)

,name,short_description,long_description,label,adjacency_list,energy,energy_unit,spin_multiplicity,optical_isomers,mass,...,harmonic_freq_3,harmonic_freq_4,harmonic_freq_5,harmonic_freq_6,harmonic_freq_7,harmonic_freq_8,harmonic_freq_9,harmonic_freq_10,harmonic_freq_11,harmonic_freq_12
id,,,,,,,,,,,,,,,,,,,,,
0,halogens_G4,B3LYP/GTBas3,,HF,"\n1 F u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-282.3080,kJ/mol,NaN,NaN,20.0062,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,halogens_G4,B3LYP/GTBas3,,HBr,"\n1 Br u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-42.7435,kJ/mol,NaN,NaN,79.9262,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,halogens_G4,B3LYP/GTBas3,,HCl,"\n1 Cl u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-99.1327,kJ/mol,NaN,NaN,35.9767,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


One important division in these data is the linear vs non-linear rotors - similar to the `thermo` databases, each of these can be selected separately by looking at their corresponding fields:

In [23]:
statmech_linear_df = df[df['linear_symmetry'].notna()].copy().dropna(axis='columns', how='all')
statmech_linear_df.head(5)


,name,short_description,long_description,label,adjacency_list,energy,energy_unit,spin_multiplicity,mass,mass_unit,...,linear_inertia_unit,linear_symmetry,harmonic_freq_unit,harmonic_freq_1,harmonic_freq_2,harmonic_freq_3,harmonic_freq_4,harmonic_freq_5,harmonic_freq_6,harmonic_freq_7
id,,,,,,,,,,,,,,,,,,,,,
0,halogens_G4,B3LYP/GTBas3,,HF,"\n1 F u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-282.30800,kJ/mol,NaN,20.0062,amu,...,amu*angstrom^2,1.0,cm^-1,4113.430,NaN,NaN,NaN,NaN,NaN,NaN
1,halogens_G4,B3LYP/GTBas3,,HBr,"\n1 Br u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-42.74350,kJ/mol,NaN,79.9262,amu,...,amu*angstrom^2,1.0,cm^-1,2635.590,NaN,NaN,NaN,NaN,NaN,NaN
2,halogens_G4,B3LYP/GTBas3,,HCl,"\n1 Cl u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-99.13270,kJ/mol,NaN,35.9767,amu,...,amu*angstrom^2,1.0,cm^-1,2956.350,NaN,NaN,NaN,NaN,NaN,NaN
3,halogens_G4,B3LYP/GTBas3,,F2,"\n1 F u0 p3 c0 {2,S}\n2 F u0 p3 c0 {1,S}\n",-5.50154,kJ/mol,NaN,37.9968,amu,...,amu*angstrom^2,2.0,cm^-1,1076.600,NaN,NaN,NaN,NaN,NaN,NaN
4,halogens_G4,B3LYP/GTBas3,,FCl,"\n1 Cl u0 p3 c0 {2,S}\n2 F u0 p3 c0 {1,S}\n",-65.04860,kJ/mol,NaN,53.9673,amu,...,amu*angstrom^2,1.0,cm^-1,788.172,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
statmech_nonlinear_df = df[df['linear_symmetry'].isna()].copy().dropna(axis='columns', how='all')
statmech_nonlinear_df.head(5)


,name,short_description,long_description,label,adjacency_list,energy,energy_unit,spin_multiplicity,optical_isomers,mass,...,harmonic_freq_3,harmonic_freq_4,harmonic_freq_5,harmonic_freq_6,harmonic_freq_7,harmonic_freq_8,harmonic_freq_9,harmonic_freq_10,harmonic_freq_11,harmonic_freq_12
id,,,,,,,,,,,,,,,,,,,,,
12,halogens_G4,B3LYP/GTBas3,,OF,"\n1 F u0 p3 c0 {2,S}\n2 O u0 p2 c0 {1,S} {3,S}...",-95.2653,kJ/mol,NaN,NaN,36.0011,...,3730.420,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,halogens_G4,B3LYP/GTBas3,,OBr,"\n1 Br u0 p3 c0 {2,S}\n2 O u0 p2 c0 {1,S} {3,...",-71.8729,kJ/mol,NaN,NaN,95.9211,...,3777.610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,halogens_G4,B3LYP/GTBas3,,OCl,"\n1 Cl u0 p3 c0 {2,S}\n2 O u0 p2 c0 {1,S} {3,...",-84.4995,kJ/mol,NaN,NaN,51.9716,...,3767.440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,halogens_G4,B3LYP/GTBas3,,FOF,"\n1 F u0 p3 c0 {3,S}\n2 F u0 p3 c0 {3,S}\n3 O ...",16.0257,kJ/mol,NaN,NaN,53.9917,...,1034.580,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,halogens_G4,B3LYP/GTBas3,,FOBr,"\n1 Br u0 p3 c0 {3,S}\n2 F u0 p3 c0 {3,S}\n3 ...",58.9402,kJ/mol,NaN,NaN,113.9120,...,907.824,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## `kinetics`

Finally, the kinetics sub-database.
This one is far and away the most complicated and the largest, so let's walk through it starting with the views:

In [28]:
kinetics_db = "data/rmgdatabase/kinetics/kinetics.db"
list_all_views(kinetics_db);

Views in database 'data/rmgdatabase/kinetics/kinetics.db':
- all_family_rules_kinetics_view
- all_library_kinetics_view
- kinetics_families_view
- kinetics_family_forbidden_groups_view
- kinetics_family_groups_view
- kinetics_family_training_dictionary_view
- kinetics_family_training_reaction_species_view
- kinetics_library_dictionary_view
- kinetics_library_reaction_species_view
- label_pairs_view


These are what each of these is [...].

Let's show how to pair up the adjacency list with the labels, as well as how to pull out all of a specific reaction type like Troe and Arrhenius.

In [ ]:
# A) SUBSET OF LIBRARIES: 
# Assemble "A + B <=> C" using the raw Adjacency Lists via SQL GROUP_CONCAT
df_kinetics_libs = pd.read_sql("""
    WITH adj_reactions AS (
        SELECT library_reaction_id,
               GROUP_CONCAT(CASE WHEN role = 'reactant' THEN adjacency_list END, '\n + \n') || 
               '\n <=> \n' ||
               GROUP_CONCAT(CASE WHEN role = 'product' THEN adjacency_list END, '\n + \n') as adjacency_reaction
        FROM kinetics_library_reaction_species_view
        GROUP BY library_reaction_id
    )
    SELECT l.library_name, l.reaction_id, ar.adjacency_reaction, l.arrhenius_type, 
           l.A_val, l.A_unit, l.n, l.Ea_val, l.Ea_unit 
    FROM all_library_kinetics_view l
    JOIN adj_reactions ar ON ar.library_reaction_id = l.reaction_id
    WHERE l.library_name IN ('C3', 'GRI-Mech3.0', 'Klippenstein_Glarborg2016')
      AND l.arrhenius_type IS NOT NULL
""", kinetics_db)
print(f"A) Loaded {len(df_kinetics_libs)} standard Arrhenius reactions from specific libraries.")
if not df_kinetics_libs.empty:
    print(df_kinetics_libs[['adjacency_reaction', 'A_val', 'Ea_val']].head(1), "\n")


--- 5. Kinetics ---
A) Loaded 505 standard Arrhenius reactions from specific libraries.
                                  adjacency_reaction  A_val  Ea_val
0  multiplicity 2\n1 C u0 p0 c0 {2,D} {5,S} {6,S}...   42.0    11.0 



In [26]:


# B) SUBSET OF FAMILIES (Training Reactions): 
# (Family *rules* apply to generic functional group nodes. For discrete molecules, we query the training reactions).
df_kinetics_fams = pd.read_sql("""
    WITH adj_reactions AS (
        SELECT training_reaction_id,
               GROUP_CONCAT(CASE WHEN role = 'reactant' THEN adjacency_list END, '\n + \n') || 
               '\n <=> \n' ||
               GROUP_CONCAT(CASE WHEN role = 'product' THEN adjacency_list END, '\n + \n') as adjacency_reaction
        FROM kinetics_family_training_reaction_species_view
        GROUP BY training_reaction_id
    )
    SELECT f.name as family_name, r.id as reaction_id, ar.adjacency_reaction,
           a.A_val, a.n, a.Ea_val
    FROM kinetics_family_training_reactions_table r
    JOIN kinetics_families_table f ON f.id = r.family_id
    JOIN adj_reactions ar ON ar.training_reaction_id = r.id
    LEFT JOIN kinetics_arrhenius_table a ON a.family_training_reaction_id = r.id
    WHERE f.name IN ('H_Abstraction', 'Disproportionation', 'R_Recombination')
      AND a.A_val IS NOT NULL
""", kinetics_db)
print(f"B) Loaded {len(df_kinetics_fams)} standard Arrhenius training reactions from specific families.")
if not df_kinetics_fams.empty:
    print(df_kinetics_fams[['adjacency_reaction', 'A_val', 'Ea_val']].head(1), "\n")



B) Loaded 3429 standard Arrhenius training reactions from specific families.
                                  adjacency_reaction  A_val  Ea_val
0  1 *1 O u0 p2 c0 {2,S} {3,S}\n2    O u0 p2 c0 {...   5.76    0.75 



In [27]:

# C) SUBSET BY SPECIFIC REACTION TYPE (e.g. ThirdBody):
df_third_body = pd.read_sql("""
    WITH adj_reactions AS (
        SELECT library_reaction_id,
               GROUP_CONCAT(CASE WHEN role = 'reactant' THEN adjacency_list END, '\n + \n') || 
               '\n <=> \n' ||
               GROUP_CONCAT(CASE WHEN role = 'product' THEN adjacency_list END, '\n + \n') as adjacency_reaction
        FROM kinetics_library_reaction_species_view
        GROUP BY library_reaction_id
    )
    SELECT l.name as library_name, ar.adjacency_reaction, 
           tb.low_A_val, tb.low_n, tb.low_Ea_val 
    FROM kinetics_third_body_table tb
    JOIN kinetics_library_reactions_table r ON r.id = tb.library_reaction_id
    JOIN kinetics_libraries_table l ON l.id = r.library_id
    JOIN adj_reactions ar ON ar.library_reaction_id = r.id
""", kinetics_db)
print(f"C) Loaded {len(df_third_body)} specific 'ThirdBody' format reactions.")
if not df_third_body.empty:
    print(df_third_body[['adjacency_reaction', 'low_A_val', 'low_Ea_val']].head(1), "\n")

C) Loaded 238 specific 'ThirdBody' format reactions.
                                  adjacency_reaction     low_A_val  low_Ea_val
0  1 H u0 p0 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n <=> \...  4.577000e+19    104400.0 

